- Removed the timestamp attribute and retained extracted temporal information.
- Created cyclic time features (hour_sin, hour_cos) to represent the periodic nature of time.
- Generated 5-minute rolling mean features for heart rate, stress, and respiration rate to capture short-term physiological trends.
- Generated 5-minute rolling standard deviation features to capture physiological variability.
- Created lag features (lag1) to incorporate information from the previous minute.
- Generated change features (hr_change, stress_change) to capture sudden physiological fluctuations.
- Applied one-hot encoding to activity type to convert categorical activities into numerical features.
- Applied one-hot encoding to sleep level to represent sleep states numerically.
- Removed rows with missing values introduced by rolling-window and lag operations.
- Performed correlation analysis to evaluate the relationship between engineered features and cognitive activity.

In [2]:
#Load Preprocessed Dataset

import pandas as pd
import numpy as np

df = pd.read_csv("../final_dfs/full_df_preprocessed.csv")

print(df.shape)

df.head()

(8557, 13)


,timestamp,stress,body_battery,respiration_rate,heart_rate,sleep_level,sleep_score,activity_type,is_cognitive,hour,hour_sin,hour_cos,day_of_week
0,2026-06-03 00:00:00,-0.450928,-1.218342,-0.341442,-0.338046,awake,NaN,unknown,0,-1.654006,0.0,1.0,2
1,2026-06-03 00:01:00,-0.450928,-1.218342,-0.341442,-0.338046,awake,NaN,unknown,0,-1.654006,0.0,1.0,2
2,2026-06-03 00:02:00,-0.098915,-1.218342,0.451435,-0.338046,awake,NaN,unknown,0,-1.654006,0.0,1.0,2
3,2026-06-03 00:03:00,-0.054913,-1.218342,0.250989,0.291760,awake,NaN,generic,0,-1.654006,0.0,1.0,2
4,2026-06-03 00:04:00,-0.010912,-1.218342,0.994868,-0.338046,awake,NaN,sedentary,0,-1.654006,0.0,1.0,2


### Handle Time

In [ ]:
# Create Temporal Features
df["hour"] = df["timestamp"].dt.hour

df["hour_sin"] = np.sin(
    2 * np.pi * df["hour"] / 24
)

df["hour_cos"] = np.cos(
    2 * np.pi * df["hour"] / 24
)

df["day_of_week"] = df["timestamp"].dt.dayofweek

# Cyclical encoding for day of week
df["dow_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["dow_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

# Drop raw temporal features
df.drop(columns=["hour", "day_of_week"], inplace=True)

### Rolling Features

In [5]:
# Rolling Mean Features

df["hr_mean_5"] = (
    df["heart_rate"]
    .rolling(window=5)
    .mean()
)

df["stress_mean_5"] = (
    df["stress"]
    .rolling(window=5)
    .mean()
)

df["resp_mean_5"] = (
    df["respiration_rate"]
    .rolling(window=5)
    .mean()
)

In [6]:
# Rolling Standard Deviation

df["hr_std_5"] = (
    df["heart_rate"]
    .rolling(window=5)
    .std()
)

df["stress_std_5"] = (
    df["stress"]
    .rolling(window=5)
    .std()
)

df["resp_std_5"] = (
    df["respiration_rate"]
    .rolling(window=5)
    .std()
)

In [7]:
# Lag Features

df["hr_lag1"] = df["heart_rate"].shift(1)

df["stress_lag1"] = df["stress"].shift(1)

df["resp_lag1"] = df["respiration_rate"].shift(1)

In [8]:
# Heart Rate Change

df["hr_change"] = (
    df["heart_rate"]
    - df["heart_rate"].shift(1)
)

In [9]:
# Stress Change

df["stress_change"] = (
    df["stress"]
    - df["stress"].shift(1)
)

In [12]:
# Remove New Missing Values

print(df.isna().sum().sum())

df = df.dropna()

print(df.shape)

3135
(5451, 29)


In [14]:
# Correlation with Target

corr = df.corr(numeric_only=True)

corr["is_cognitive"] \
    .sort_values(ascending=False)

is_cognitive               1.000000
hour_cos                   0.409697
resp_std_5                 0.283560
body_battery               0.281509
stress_mean_5              0.120621
stress                     0.116829
activity_type_generic      0.115265
stress_lag1                0.107799
activity_type_sedentary    0.070621
resp_mean_5                0.046834
respiration_rate           0.044879
resp_lag1                  0.039873
stress_change              0.012561
sleep_score               -0.019221
hr_change                 -0.028908
activity_type_running     -0.041146
day_of_week               -0.110383
activity_type_walking     -0.128639
activity_type_unknown     -0.135502
hr_std_5                  -0.173617
stress_std_5              -0.173924
hour_sin                  -0.343165
hour                      -0.345124
hr_lag1                   -0.348468
hr_mean_5                 -0.354955
heart_rate                -0.359197
sleep_level_deep                NaN
sleep_level_light           

In [15]:
df.to_csv(
    "../final_dfs/full_df_features.csv",
    index=False
)

print("Feature engineered dataset saved.")

Feature engineered dataset saved.
